<a href="https://colab.research.google.com/github/Cristianriosrivas/Cristianriosrivas/blob/main/Agente_LuminaStore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain langchain-classic langchain-community langchain-core langchain-cohere langchain-text-splitters faiss-cpu pypdf

In [3]:
import getpass
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings, ChatCohere
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain # <- AQUÍ ESTÁ LA HERRAMIENTA FALTANTE
from langchain_core.prompts import ChatPromptTemplate

# 1. Cargamos el PDF
print("1/4 - Leyendo y preparando el documento PDF...")
loader = PyPDFLoader("documento_luminastore.pdf")
paginas = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
fragmentos = text_splitter.split_documents(paginas)

# 2. Clave de Cohere
print("\n2/4 - Autenticación")
os.environ["COHERE_API_KEY"] = getpass.getpass("Por favor, pega tu API Key de Cohere y presiona Enter: ")

# 3. Creando el 'cerebro' del agente (Versión Cohere)
print("\n3/4 - Creando la base de datos y conectando a Cohere...")
embeddings = CohereEmbeddings(model="embed-multilingual-v3.0")
vectorstore = FAISS.from_documents(fragmentos, embeddings)
retriever = vectorstore.as_retriever()
llm = ChatCohere(model="command-a-03-2025")

# 4. Instrucciones del Agente
instrucciones = (
    "Eres un asistente virtual experto en soporte al cliente para LuminaStore. "
    "Usa EXCLUSIVAMENTE los siguientes fragmentos de información para responder. "
    "Si la respuesta no está en el texto, di amablemente que no tienes esa información. "
    "Responde de forma profesional, clara y directa.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", instrucciones),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
agente_luminastore = create_retrieval_chain(retriever, question_answer_chain)

# 5. La gran prueba
print("\n4/4 - ¡Todo listo! Consultando a la IA...")
pregunta_cliente = "¿Cuánto cuesta el envío estándar y qué pasa si mi código postal es de difícil acceso?"

respuesta = agente_luminastore.invoke({"input": pregunta_cliente})

print("\n==================================================")
print("🗣️ RESPUESTA DE TU AGENTE LUMINASTORE (COHERE):")
print(respuesta["answer"])
print("==================================================")

1/4 - Leyendo y preparando el documento PDF...

2/4 - Autenticación
Por favor, pega tu API Key de Cohere y presiona Enter: ··········

3/4 - Creando la base de datos y conectando a Cohere...

4/4 - ¡Todo listo! Consultando a la IA...

🗣️ RESPUESTA DE TU AGENTE LUMINASTORE (COHERE):
El costo del envío estándar es de **$6.50 USD** y tiene un tiempo de tránsito de 4 a 7 días hábiles. Si tu código postal es considerado como "Zona Extendida" (áreas rurales, fronterizas o de difícil acceso logístico), se aplicará un recargo fijo de **$5.00 USD** al costo del envío, independientemente de si la orden califica para envío gratuito. Además, el tiempo de tránsito puede extenderse entre 3 y 5 días hábiles adicionales a los tiempos estándar.
